# Criação da tabela Gold - JOGADORES

## Objetivos

- Agrupamento de estatísticas gerais por jogador

In [0]:
from pyspark.sql.functions import *

In [0]:
df_cards_player = (
    spark.read.table("api_football.silver.cards")
    # agrupamento devido agregação (soma)
    .groupBy("team_id","team_name","player_name")
    .agg(
        # soma de cartões amarelos por time: quando a linha da coluna card_type está como "yellow card" soma um, caso contrário soma zero
        sum(when(col("card_type") == "yellow card", 1).otherwise(0)).alias( "yellow_cards"),
        # soma de cartões vermelhos por time (mesmo processo anterior)
        sum(when(col("card_type") == "red card", 1).otherwise(0)).alias("red_cards"),
    )
)

#display(df_cards_player)

In [0]:
df_goals_player = (
    spark.read.table("api_football.silver.goals")
    .filter(~col("player_name").like("%(o.g.)%"))
    # agrupamento devido agregação
    .groupBy("team_id", "team_name", "player_name")
    # contagem de gols por jogador e time
    .agg(count("*").alias("player_goals"))
    # ordenação por gols por jogador
).orderBy(col("player_goals").desc())

#display(df_goals_player)

In [0]:
df_assists_player = (
    spark.read.table("api_football.silver.goals")
    # filtragem de linhas onde assist_name não é nulo e diferente de vazio
    .filter(col("assist_name").isNotNull() & (trim(col("assist_name")) != ""))
    # agrupamento devido agregação e ajuste de nome de coluna
    .groupBy("team_id", "team_name", col("assist_name").alias("player_name"))
    # contagem de assistências por jogador e time
    .agg(count("*").alias("player_assists"))
    # ordenação por assistências por jogador
).orderBy(col("player_assists").desc())

#display(df_assists_player)

In [0]:
df_gold_player = (
    df_goals_player.alias("g")
    # full join para pegar todos os dados, visto que nem todos jogadores vão aparecer nos três DataFrames
    .join(df_assists_player.alias("a"), ["team_id", "team_name", "player_name"], "full")
    .join(df_cards_player.alias("c"), ["team_id", "team_name", "player_name"], "full")
    #seleção de colunas
    .select(
        # seleciona o primeiro valor não nulo da lista de valores e armazena na coluna (unifica as colunas iguais provenientes de diferentes DataFrames)
        coalesce(col("g.team_id"), col("a.team_id"), col("c.team_id")).alias("team_id"),
        coalesce(col("g.team_name"), col("a.team_name"), col("c.team_name")).alias("team_name"),
        coalesce(col("g.player_name"), col("a.player_name"), col("c.player_name")).alias("player_name"),
        # se a coluna for nula, substitui por zero
        coalesce(col("g.player_goals"), lit(0)).alias("goals"),
        coalesce(col("a.player_assists"), lit(0)).alias("assists"),
        coalesce(col("c.yellow_cards"), lit(0)).alias("yellow_cards"),
        coalesce(col("c.red_cards"), lit(0)).alias("red_cards")
    )
# ordenação por gols e assistências
).orderBy(col("goals").desc(), col("assists").desc())

display(df_gold_player)

In [0]:
df_gold_player.write.mode("overwrite").saveAsTable("api_football.gold.players_statistics")

In [0]:
%sql
-- select
--   *
-- from
--   api_football.gold.players_statistics